# Spam Email Detection with NLP
## Notebook 1 — Project Overview and Data Inspection
**Author: Tinashe Clifford Makandise**

## 1. Problem definition
Build a Python application that predicts whether supplied email text resembles labelled spam or ham. The first version is an English-focused historical benchmark demonstration. Spam is the positive class (`1`); ham is the negative class (`0`). Predicted ham is not proof an email is safe. This is not a phishing, attachment or sender-reputation scanner.

### Objectives
Inspect and validate data; create a reproducible holdout; build text-processing pipelines; compare a baseline, Multinomial Naive Bayes and Logistic Regression; evaluate errors; and deploy a Streamlit application. Compare additional models only if justified.

### Evaluation plan
Use spam-class F1 as the primary model-selection metric, with precision, recall, false-positive rate, confusion matrix and average precision as supporting evidence. Accuracy is secondary because most messages are ham. Any threshold selection or calibration will use training/validation data only. Fit TF-IDF inside cross-validation pipelines. Preserve the test partition for Notebook 5. Report actual latency and measured performance rather than assume 97% accuracy or one-second responses.

## 2. Dataset, attribution and setup
Source: MIT OpenCourseWare, *The Analytics Edge*, Spring 2017, Assignment 5: Separating Spam from Ham. MIT describes ham from Vincent Kaminski's Enron inbox and spam from SpamAssassin and Project Honey Pot, referring to Metsis, Androutsopoulos and Paliouras (2006), *Spam Filtering with Naive Bayes — Which Naive Bayes?*

- [Dataset page](https://ocw.mit.edu/courses/15-071-the-analytics-edge-spring-2017/resources/emails/)
- [Assignment and provenance](https://ocw.mit.edu/courses/15-071-the-analytics-edge-spring-2017/pages/text-analytics/assignment-5/)
- [Download CSV](https://ocw.mit.edu/courses/15-071-the-analytics-edge-spring-2017/67f7be241f1b459d7fdb8e9031384338_emails.csv)

Save the CSV unchanged as `spam-email-detector/data/raw/emails.csv`. Save this notebook in `spam-email-detector/notebooks/`. Launch Jupyter from the project root or `notebooks/`. If needed, run `%pip install pandas numpy scikit-learn` once and restart the kernel.

No email examples are printed or embedded in notebook output. Keep email contents and generated data partitions out of the public repository; provide source download instructions and attribution. Public availability does not establish unrestricted redistribution permission.

In [1]:
from pathlib import Path
import hashlib, json, platform, re, unicodedata
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data/raw/emails.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
if not RAW_PATH.is_file():
    raise FileNotFoundError(f"Save the unchanged MIT emails.csv at {RAW_PATH}")
for folder in [REPORTS_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version(), "pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)


Python: 3.12.14 pandas: 2.2.3 scikit-learn: 1.8.0


## 3. Validate the raw file
Inspect structure, not word frequencies or predictive relationships, before the split. The expected original file contains 5,728 rows and two columns: `text` and `spam`. Invalid or changed sources stop execution instead of silently filling labels. A source hash records exactly which bytes were used. Row IDs refer to original zero-based CSV row positions, not a model input.

In [2]:
raw = pd.read_csv(RAW_PATH)
assert raw.shape == (5728, 2), f"Unexpected shape: {raw.shape}"
assert set(raw.columns) == {"text", "spam"}
assert raw["spam"].notna().all() and raw["spam"].isin([0, 1]).all()
assert raw["text"].notna().all()
assert raw["text"].map(lambda value: isinstance(value, str)).all()
assert raw["text"].str.strip().ne("").all()
raw["spam"] = raw["spam"].astype(int)
raw.insert(0, "row_id", np.arange(len(raw)))
source_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print("Rows:", len(raw), "Original columns: text, spam")
print("Missing fields:", raw[["text", "spam"]].isna().sum().to_dict())
print("Source SHA-256:", source_hash)
print("Source label counts:", raw["spam"].value_counts().sort_index().to_dict())

Rows: 5728 Original columns: text, spam
Missing fields: {'text': 0, 'spam': 0}
Source SHA-256: a5d0d75d15d370ef2dd3229b30204b18deab6d3fd249206e89a2c58f901bcc77
Source label counts: {0: 4360, 1: 1368}


## 4. Duplicate audit
Repeated emails can inflate performance if copies land in both training and test sets. First remove exact `(text, label)` repetitions, retaining the first occurrence. Then define a grouping key using Unicode NFKC normalisation, case folding and collapsed whitespace. This key is used only for integrity and partitioning; model input text remains unchanged.

Normalised groups with contradictory labels are quarantined in full rather than choosing a label arbitrarily. Such rows are excluded before splitting and their IDs are recorded. Normalised variants with consistent labels remain in the data but must stay in the same partition and later in the same cross-validation fold.

This protects against exact and case/whitespace-equivalent duplicates. It does **not** guarantee separation of near-duplicate campaigns or email threads. A stronger similarity audit may be needed; any future grouping changes require a newly documented split before model development.

In [3]:
exact_duplicate_mask = raw.duplicated(subset=["text", "spam"], keep="first")
exact_duplicate_ids = raw.loc[exact_duplicate_mask, "row_id"].tolist()
unique = raw.loc[~exact_duplicate_mask].copy()
def normalise_for_grouping(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text).casefold()).strip()
unique["group_id"] = unique["text"].map(
    lambda text: hashlib.sha256(normalise_for_grouping(text).encode("utf-8")).hexdigest())
label_counts_per_group = unique.groupby("group_id")["spam"].nunique()
conflict_groups = set(label_counts_per_group[label_counts_per_group > 1].index)
conflict_mask = unique["group_id"].isin(conflict_groups)
quarantined_ids = unique.loc[conflict_mask, "row_id"].tolist()
eligible = unique.loc[~conflict_mask].copy()
assert eligible.groupby("group_id")["spam"].nunique().eq(1).all()
print("Exact duplicate rows removed:", len(exact_duplicate_ids))
print("Conflicting normalised groups:", len(conflict_groups))
print("Rows quarantined for conflicting labels:", len(quarantined_ids))
print("Eligible rows:", len(eligible), "Distinct normalised groups:", eligible["group_id"].nunique())

Exact duplicate rows removed: 33
Conflicting normalised groups: 0
Rows quarantined for conflicting labels: 0
Eligible rows: 5695 Distinct normalised groups: 5532


## 5. Reserve a stratified group holdout
Split unique groups, stratifying by their unambiguous labels. Select 20% of groups for testing with seed 42, then recover all rows belonging to each group. Row proportions may differ slightly from 80/20 because groups vary in size. Do not search multiple random seeds for better performance.

This is a random within-corpus evaluation, not a future-time or new-organisation evaluation. Ham messages from one corporate inbox may introduce source-specific vocabulary; strong benchmark results will not establish general email filtering performance.

In [4]:
groups = eligible[["group_id", "spam"]].drop_duplicates("group_id").sort_values("group_id")
train_groups, test_groups = train_test_split(groups, test_size=.20, random_state=42, stratify=groups["spam"])
train = eligible[eligible["group_id"].isin(train_groups["group_id"])].sort_values("row_id").copy()
test = eligible[eligible["group_id"].isin(test_groups["group_id"])].sort_values("row_id").copy()
assert set(train["group_id"]).isdisjoint(test["group_id"])
assert set(train["row_id"]).isdisjoint(test["row_id"])
assert len(train) + len(test) == len(eligible)
assert set(train["spam"]) == {0, 1} and set(test["spam"]) == {0, 1}
assert not train[["text", "spam"]].duplicated().any()
print(f"Training: {len(train):,} rows; {train['group_id'].nunique():,} groups")
print(f"Reserved test: {len(test):,} rows; {test['group_id'].nunique():,} groups")
print("Verified zero group overlap between partitions.")

Training: 4,547 rows; 4,425 groups
Reserved test: 1,148 rows; 1,107 groups
Verified zero group overlap between partitions.


## 6. Training-only initial inspection
Detailed word exploration belongs in Notebook 2. Here we summarise training class balance, text-length coverage and the presence of a leading `Subject:` prefix. The provided CSV is already a text representation rather than a full raw MIME-email archive. Do not assume original HTML, punctuation, attachment or sender information is available. The future app needs the same deterministic input handling as the trained pipeline.

A majority-class predictor can look strong on accuracy while detecting no spam. Model comparison will include a formal training-fitted baseline. No vectoriser or classifier is fitted here.

In [5]:
class_summary = train["spam"].value_counts().sort_index().rename("count").to_frame()
class_summary["label"] = class_summary.index.map({0: "ham", 1: "spam"})
class_summary["percent"] = class_summary["count"] / len(train) * 100
print(class_summary.to_string(index=True))
length_summary = train["text"].str.len().describe().to_frame("training_character_count")
print(length_summary.to_string())
subject_prefix_rate = float(train["text"].str.match(r"(?i)^\s*subject\s*:").mean())
print(f"Training texts starting with Subject: {subject_prefix_rate:.1%}")
print(f"Training majority share: {train['spam'].value_counts(normalize=True).max():.1%}")

      count label   percent
spam                       
0      3453   ham  75.94018
1      1094  spam  24.05982
       training_character_count
count               4547.000000
mean                1587.229382
std                 2155.318014
min                   13.000000
25%                  515.000000
50%                  984.000000
75%                 1903.500000
max                43952.000000
Training texts starting with Subject: 100.0%
Training majority share: 75.9%


## 7. Export reproducible handover files
Exports contain unchanged eligible text, numeric label, source row ID and duplicate-group ID. In later notebooks, features are the `text` column and target is `spam`; IDs must never be predictors. Group-aware cross-validation must use `group_id` to keep retained variants together.

The manifest records provenance, split membership and exclusions. These local data outputs contain email text and should not be uploaded to GitHub. Existing generated files are overwritten on rerun using the same source and rules.

In [6]:
for name, frame in [("train", train), ("test", test)]:
    frame[["row_id", "text", "spam", "group_id"]].to_csv(PROCESSED_DIR / f"{name}.csv", index=False)
class_summary.to_csv(REPORTS_DIR / "training_class_summary.csv", index_label="spam")
length_summary.to_csv(REPORTS_DIR / "training_length_summary.csv")
manifest = {
    "author": "Tinashe Clifford Makandise",
    "source_url": "https://ocw.mit.edu/courses/15-071-the-analytics-edge-spring-2017/67f7be241f1b459d7fdb8e9031384338_emails.csv",
    "source_sha256": source_hash, "source_rows": len(raw),
    "label_mapping": {"0": "ham", "1": "spam"},
    "exact_duplicate_ids_removed": exact_duplicate_ids,
    "conflicting_label_ids_quarantined": quarantined_ids,
    "grouping": "SHA256 of NFKC + casefold + whitespace collapse; input text preserved",
    "split": "stratified split of normalised groups; 20% groups held out; seed 42",
    "train_rows": len(train), "test_rows": len(test),
    "train_row_ids": train["row_id"].astype(int).tolist(), "test_row_ids": test["row_id"].astype(int).tolist(),
    "train_groups": sorted(train["group_id"].unique().tolist()), "test_groups": sorted(test["group_id"].unique().tolist()),
    "primary_selection_metric": "spam-class F1", "cv_requirement": "stratified group-aware CV on training partition",
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__},
}
for name in ["train", "test"]:
    manifest[f"{name}_file_sha256"] = hashlib.sha256((PROCESSED_DIR / f"{name}.csv").read_bytes()).hexdigest()
(REPORTS_DIR / "data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
# Verify saved text and label values survive CSV round trips, including multiline content.
for name, original in [("train", train), ("test", test)]:
    restored = pd.read_csv(PROCESSED_DIR / f"{name}.csv", keep_default_na=False)
    assert restored["text"].tolist() == original["text"].tolist()
    assert restored["spam"].tolist() == original["spam"].tolist()
print("Partitions and manifest saved; CSV round-trip checks passed.")

Partitions and manifest saved; CSV round-trip checks passed.


## 8. Findings and progression
Source validation and duplicate integrity checks are complete. A fixed group-separated holdout is reserved; training-only class and length summaries are available. No performance result has been estimated and no text transformation has been fitted.

**Notebook 2:** explore training message lengths and vocabulary, inspect source-specific patterns, and document preprocessing candidates. Subsequent notebooks will compare minimal cleaning with selected alternatives, fit TF-IDF inside group-aware cross-validation, choose settings using training data and reserve the holdout for final evaluation.

Keep source and partition files local. GitHub deliverables should include code, explanations, aggregate evidence and source links, with raw message excerpts omitted from saved outputs.